# CondFD Sky Longwave Radiation Override

Override the sky longwave radiation term (`hsky * (Tsky - Tsurf)`) on CondFD exterior surfaces
with a user-supplied net sky radiation flux (W/m²). Works with **any IDF/EPW pair**.

### What this does

EnergyPlus normally computes sky LW radiation using a linearized h-coefficient model.
This actuator replaces that calculation with your own value — from COMSOL, pyrgeometer data,
or any other source.

### Sign convention

| Enet value | Meaning |
|---|---|
| **Negative** (e.g. -200) | Surface loses heat to sky (typical nighttime cooling) |
| **Zero** | No sky LW exchange |
| **Positive** (e.g. +200) | Sky heats the surface (unusual but valid) |

### Requirements

- EnergyPlus build with Python API support
- IDF must use `HeatBalanceAlgorithm: ConductionFiniteDifference` (globally or per-surface)
- `pandas` and `matplotlib` for results display

---
## Step 1: Configuration

Edit the three paths below. Everything else is auto-detected.

In [ ]:
# ============================================================
# EDIT THESE THREE PATHS
# ============================================================

ENERGYPLUS_DIR = r"C:\projects\EnergyPlus\build\Products\Release"
IDF_FILE       = r"C:\projects\EnergyPlus\testfiles\1ZoneCondFD_Enet_Test.idf"
EPW_FILE       = r"C:\projects\EnergyPlus\weather\USA_CO_Golden-NREL.724666_TMY3.epw"

---
## Step 2: Parse IDF and find eligible surfaces

This cell reads the IDF text and identifies all exterior CondFD surfaces that can be overridden.
A surface is eligible if:
1. It is a `BuildingSurface:Detailed` (or `Wall:Detailed`, `RoofCeiling:Detailed`, `Floor:Detailed`)
2. Its outside boundary condition is `Outdoors`
3. The heat balance algorithm is `ConductionFiniteDifference` (globally or per-surface)

In [ ]:
import re
from pathlib import Path

def parse_idf_objects(idf_text):
    """Parse IDF text into a list of (object_type, [field_values]) tuples."""
    # Strip comments
    lines = []
    for line in idf_text.splitlines():
        # Remove inline comments (everything after ! outside of strings)
        idx = line.find("!")
        if idx >= 0:
            line = line[:idx]
        lines.append(line.strip())
    text = " ".join(lines)

    # Split on semicolons to get objects
    objects = []
    for obj_str in text.split(";"):
        obj_str = obj_str.strip()
        if not obj_str:
            continue
        fields = [f.strip() for f in obj_str.split(",")]
        if fields:
            obj_type = fields[0]
            obj_fields = fields[1:]
            objects.append((obj_type, obj_fields))
    return objects


def find_condfd_exterior_surfaces(idf_path):
    """Find all exterior surfaces that use CondFD heat balance.

    Returns list of dicts: {name, surface_type, construction, zone, boundary_cond}
    """
    idf_text = Path(idf_path).read_text(encoding="utf-8", errors="replace")
    objects = parse_idf_objects(idf_text)

    # Check global heat balance algorithm
    global_condfd = False
    for obj_type, fields in objects:
        if obj_type.lower() == "heatbalancealgorithm" and fields:
            if "conductionfinitedifference" in fields[0].lower():
                global_condfd = True
            break

    # Collect per-surface algorithm overrides
    # SurfaceProperty:HeatTransferAlgorithm, <surface>, <algorithm>
    per_surface_algo = {}  # surface_name_upper -> algorithm
    # SurfaceProperty:HeatTransferAlgorithm:SurfaceList, <list_name>, <algorithm>, surf1, surf2, ...
    for obj_type, fields in objects:
        ot = obj_type.lower()
        if ot == "surfaceproperty:heattransferalgorithm" and len(fields) >= 2:
            per_surface_algo[fields[0].upper()] = fields[1].lower()
        elif ot == "surfaceproperty:heattransferalgorithm:surfacelist" and len(fields) >= 3:
            algo = fields[1].lower()
            for surf_name in fields[2:]:
                if surf_name:
                    per_surface_algo[surf_name.upper()] = algo
        elif ot == "surfaceproperty:heattransferalgorithm:construction" and len(fields) >= 2:
            # construction-based override — we'd need to map constructions to surfaces
            # For simplicity, store with a prefix
            pass  # handled below

    # Collect construction-based overrides
    condfd_constructions = set()  # construction names that force CondFD
    for obj_type, fields in objects:
        ot = obj_type.lower()
        if ot == "surfaceproperty:heattransferalgorithm:construction" and len(fields) >= 2:
            algo = fields[1].lower()
            if "conductionfinitedifference" in algo:
                constr_name = fields[2].upper() if len(fields) > 2 else ""
                if constr_name:
                    condfd_constructions.add(constr_name)

    # Parse surfaces
    surface_types = {
        "buildingsurface:detailed",
        "wall:detailed", "roofceiling:detailed", "floor:detailed",
        "wall:exterior", "roof",
    }

    surfaces = []
    for obj_type, fields in objects:
        ot = obj_type.lower()
        if ot not in surface_types:
            continue

        if ot == "buildingsurface:detailed" and len(fields) >= 5:
            name = fields[0]
            stype = fields[1]
            construction = fields[2]
            zone = fields[3]
            bc = fields[5] if len(fields) > 5 else ""
        elif ot in ("wall:detailed", "roofceiling:detailed", "floor:detailed") and len(fields) >= 4:
            name = fields[0]
            stype = ot.split(":")[0].capitalize()
            construction = fields[1]
            zone = fields[2]
            bc = fields[4] if len(fields) > 4 else ""
        elif ot == "wall:exterior" and len(fields) >= 3:
            name = fields[0]
            stype = "Wall"
            construction = fields[1]
            zone = fields[2] if len(fields) > 2 else ""
            bc = "Outdoors"  # always exterior
        elif ot == "roof" and len(fields) >= 3:
            name = fields[0]
            stype = "Roof"
            construction = fields[1]
            zone = fields[2] if len(fields) > 2 else ""
            bc = "Outdoors"  # always exterior
        else:
            continue

        # Check exterior
        if bc.lower() != "outdoors":
            continue

        # Check CondFD: per-surface override > construction override > global
        name_upper = name.upper()
        if name_upper in per_surface_algo:
            is_condfd = "conductionfinitedifference" in per_surface_algo[name_upper]
        elif construction.upper() in condfd_constructions:
            is_condfd = True
        else:
            is_condfd = global_condfd

        if not is_condfd:
            continue

        surfaces.append({
            "name": name,
            "type": stype,
            "construction": construction,
            "zone": zone,
        })

    return surfaces


# --- Run detection ---
eligible = find_condfd_exterior_surfaces(IDF_FILE)

if not eligible:
    print("ERROR: No exterior CondFD surfaces found in this IDF.")
    print("  Check that HeatBalanceAlgorithm is set to ConductionFiniteDifference")
    print("  and the IDF has exterior surfaces (Outside Boundary Condition = Outdoors).")
else:
    print(f"Found {len(eligible)} eligible surface(s):\n")
    for i, s in enumerate(eligible):
        print(f"  [{i}] {s['name']:30s}  {s['type']:6s}  construction={s['construction']}  zone={s['zone']}")

---
## Step 3: Choose surfaces and Enet values

Pick which surfaces to override and what Enet values to test.
A baseline run (no override) is always included automatically.

In [ ]:
# Which surfaces to override?
# Options:
#   "all"              - override all eligible surfaces found above
#   [0, 2]             - override surfaces by index (from the list above)
#   ["Zn001:Roof001"]  - override surfaces by name

SURFACES = "all"

# Enet values to test (W/m2). Baseline (no override) is always included.
# Negative = cooling, positive = heating, 0 = no sky exchange.
ENET_VALUES = [0, -200]

# Design-day only? Set False for annual run (slower).
DESIGN_DAY_ONLY = True

# ============================================================
# Resolve surface selection
# ============================================================
if SURFACES == "all":
    target_surfaces = [s["name"] for s in eligible]
elif isinstance(SURFACES, list) and len(SURFACES) > 0:
    if isinstance(SURFACES[0], int):
        target_surfaces = [eligible[i]["name"] for i in SURFACES]
    else:
        target_surfaces = list(SURFACES)
else:
    target_surfaces = [s["name"] for s in eligible]

# Ensure baseline is first
run_configs = [None] + [e for e in ENET_VALUES if e is not None]

print(f"Surfaces to override: {target_surfaces}")
print(f"Runs: baseline + Enet = {[e for e in ENET_VALUES]}")
print(f"Total simulations: {len(run_configs)}")

---
## Step 4: Run simulations

Runs EnergyPlus once per Enet value (plus baseline). Data is collected via API callbacks.

In [ ]:
import sys
from pathlib import Path
from tempfile import mkdtemp

ep_path = str(Path(ENERGYPLUS_DIR).resolve())
if ep_path not in sys.path:
    sys.path.insert(0, ep_path)

from pyenergyplus.api import EnergyPlusAPI


def run_single(idf_file, epw_file, target_surfaces, enet_value, design_day_only=True):
    """Run one EnergyPlus simulation, optionally overriding sky LW on target surfaces.

    Args:
        enet_value: W/m2 override, or None for baseline (no override).

    Returns:
        list of dicts with timestep data.
    """
    _api = EnergyPlusAPI()
    _state = _api.state_manager.new_state()
    label = f"Enet={enet_value}" if enet_value is not None else "baseline"
    _run_dir = mkdtemp(prefix=f"enet_{label}_")
    _data = []

    def _make_callback():
        ready = [False]
        act_handles = [{}]   # surface_name -> handle
        var_handles = [{}]   # surface_name -> {surf_temp, lw_rad}
        global_vh = [{}]     # oat, sky_temp

        def _cb(st):
            if not ready[0]:
                if not _api.exchange.api_data_fully_ready(st):
                    return

                for surf in target_surfaces:
                    h = _api.exchange.get_actuator_handle(
                        st, "CondFD Surface", "Sky Longwave Radiation Override", surf
                    )
                    if h == -1:
                        print(f"  WARNING: actuator not found for '{surf}' (not CondFD exterior?)")
                        continue
                    act_handles[0][surf] = h
                    var_handles[0][surf] = {
                        "surf_temp": _api.exchange.get_variable_handle(
                            st, "Surface Outside Face Temperature", surf),
                        "lw_rad": _api.exchange.get_variable_handle(
                            st, "Surface Outside Face Net Thermal Radiation Heat Gain Rate per Area", surf),
                    }

                global_vh[0] = {
                    "oat": _api.exchange.get_variable_handle(
                        st, "Site Outdoor Air Drybulb Temperature", "Environment"),
                    "sky_temp": _api.exchange.get_variable_handle(
                        st, "Site Sky Temperature", "Environment"),
                }
                ready[0] = True

            # Apply override
            if enet_value is not None:
                for surf, h in act_handles[0].items():
                    _api.exchange.set_actuator_value(st, h, enet_value)

            # Skip warmup data
            if _api.exchange.warmup_flag(st):
                return

            # Collect
            oat = _api.exchange.get_variable_value(st, global_vh[0]["oat"])
            sky_t = _api.exchange.get_variable_value(st, global_vh[0]["sky_temp"])
            hour = _api.exchange.hour(st) + _api.exchange.minutes(st) / 60.0
            month = _api.exchange.month(st)
            day = _api.exchange.day_of_month(st)

            for surf in act_handles[0]:
                vh = var_handles[0][surf]
                _data.append({
                    "month": month, "day": day, "hour": hour,
                    "surface": surf,
                    "oat_C": oat, "sky_temp_C": sky_t,
                    "surf_temp_C": _api.exchange.get_variable_value(st, vh["surf_temp"]),
                    "lw_rad_Wm2": _api.exchange.get_variable_value(st, vh["lw_rad"]),
                    "run": label,
                })

        return _cb

    _api.runtime.callback_begin_zone_timestep_after_init_heat_balance(_state, _make_callback())

    args = ["-d", _run_dir, "-w", epw_file, idf_file]
    if design_day_only:
        args = ["-D"] + args

    print(f"  Running {label}...", end=" ", flush=True)
    ret = _api.runtime.run_energyplus(_state, args)
    _api.state_manager.delete_state(_state)

    if ret == 0:
        print(f"OK ({len(_data)} data points)")
    else:
        print(f"FAILED (return code {ret}). Check {_run_dir}/eplusout.err")

    return _data


# --- Run all configurations ---
print(f"Running {len(run_configs)} simulation(s)...\n")
all_data = []
for enet in run_configs:
    all_data.extend(run_single(IDF_FILE, EPW_FILE, target_surfaces, enet, DESIGN_DAY_ONLY))

print(f"\nDone. Total data points: {len(all_data)}")

---
## Step 5: Results

In [ ]:
import pandas as pd

df = pd.DataFrame(all_data)

if len(df) == 0:
    print("No data collected. Check the error messages above.")
else:
    # Summary per run
    summary = df.groupby(["run", "surface"]).agg(
        min_surf_temp=("surf_temp_C", "min"),
        max_surf_temp=("surf_temp_C", "max"),
        mean_surf_temp=("surf_temp_C", "mean"),
        mean_lw_rad=("lw_rad_Wm2", "mean"),
    ).round(1)

    print("Summary:")
    display(summary)
    print()
    display(df.head(10))

In [ ]:
import matplotlib.pyplot as plt

if len(df) > 0:
    surfaces = df["surface"].unique()
    runs = df["run"].unique()

    for surf in surfaces:
        sdf = df[df["surface"] == surf]

        fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

        # --- Top: Surface temperature comparison ---
        ax1 = axes[0]
        for run_label in runs:
            rdf = sdf[sdf["run"] == run_label].reset_index(drop=True)
            ax1.plot(rdf["hour"], rdf["surf_temp_C"], label=run_label, linewidth=1.5)

        # Add OAT and sky temp from baseline for reference
        bdf = sdf[sdf["run"] == "baseline"].reset_index(drop=True)
        if len(bdf) > 0:
            ax1.plot(bdf["hour"], bdf["oat_C"], label="OAT", linewidth=1, linestyle="--", color="gray", alpha=0.6)
            ax1.plot(bdf["hour"], bdf["sky_temp_C"], label="Sky Temp", linewidth=1, linestyle=":", color="gray", alpha=0.6)

        ax1.set_ylabel("Temperature (°C)")
        ax1.set_title(f"{surf} — Surface Temperature")
        ax1.legend(loc="best", fontsize=9)
        ax1.grid(True, alpha=0.3)

        # --- Bottom: LW radiation comparison ---
        ax2 = axes[1]
        for run_label in runs:
            rdf = sdf[sdf["run"] == run_label].reset_index(drop=True)
            ax2.plot(rdf["hour"], rdf["lw_rad_Wm2"], label=run_label, linewidth=1.5)

        ax2.set_ylabel("LW Radiation (W/m²)")
        ax2.set_xlabel("Hour of Day")
        ax2.set_title(f"{surf} — Net Thermal Radiation Heat Gain")
        ax2.legend(loc="best", fontsize=9)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
else:
    print("No data to plot.")

---
## Step 6: Export results (optional)

Save the full dataset to CSV for further analysis.

In [ ]:
if len(df) > 0:
    out_csv = Path(IDF_FILE).stem + "_enet_results.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved to {out_csv}")

---
## Advanced: Time-varying Enet from CSV

Instead of a constant Enet, you can supply a time-series (e.g. from COMSOL or pyrgeometer data).

Prepare a CSV with columns `month`, `day`, `hour`, `enet_Wm2`. The callback will look up the
nearest timestep and apply that value.

Example CSV:
```
month,day,hour,enet_Wm2
7,21,0.0,-180
7,21,1.0,-190
7,21,2.0,-200
...
```

In [ ]:
def run_with_csv_enet(idf_file, epw_file, target_surfaces, enet_csv_path, design_day_only=True):
    """Run EnergyPlus with time-varying Enet from a CSV file.

    CSV must have columns: month, day, hour, enet_Wm2
    The hour column is fractional (e.g. 1.5 = 1:30).
    Values are linearly interpolated between rows.
    """
    import numpy as np

    enet_schedule = pd.read_csv(enet_csv_path)
    required_cols = {"month", "day", "hour", "enet_Wm2"}
    if not required_cols.issubset(enet_schedule.columns):
        print(f"ERROR: CSV must have columns {required_cols}, got {set(enet_schedule.columns)}")
        return []

    # Build lookup: convert to sequential hours for interpolation
    enet_schedule["seq_hour"] = (
        (enet_schedule["month"] - 1) * 744  # approximate
        + (enet_schedule["day"] - 1) * 24
        + enet_schedule["hour"]
    )
    seq_hours = enet_schedule["seq_hour"].values
    enet_vals = enet_schedule["enet_Wm2"].values

    _api = EnergyPlusAPI()
    _state = _api.state_manager.new_state()
    _run_dir = mkdtemp(prefix="enet_csv_")
    _data = []

    def _make_callback():
        ready = [False]
        act_handles = [{}]
        var_handles = [{}]
        global_vh = [{}]

        def _cb(st):
            if not ready[0]:
                if not _api.exchange.api_data_fully_ready(st):
                    return
                for surf in target_surfaces:
                    h = _api.exchange.get_actuator_handle(
                        st, "CondFD Surface", "Sky Longwave Radiation Override", surf)
                    if h != -1:
                        act_handles[0][surf] = h
                        var_handles[0][surf] = {
                            "surf_temp": _api.exchange.get_variable_handle(
                                st, "Surface Outside Face Temperature", surf),
                            "lw_rad": _api.exchange.get_variable_handle(
                                st, "Surface Outside Face Net Thermal Radiation Heat Gain Rate per Area", surf),
                        }
                global_vh[0] = {
                    "oat": _api.exchange.get_variable_handle(
                        st, "Site Outdoor Air Drybulb Temperature", "Environment"),
                    "sky_temp": _api.exchange.get_variable_handle(
                        st, "Site Sky Temperature", "Environment"),
                }
                ready[0] = True

            # Interpolate Enet for current timestep
            month = _api.exchange.month(st)
            day = _api.exchange.day_of_month(st)
            hour = _api.exchange.hour(st) + _api.exchange.minutes(st) / 60.0
            current_seq = (month - 1) * 744 + (day - 1) * 24 + hour
            enet_now = float(np.interp(current_seq, seq_hours, enet_vals))

            for surf, h in act_handles[0].items():
                _api.exchange.set_actuator_value(st, h, enet_now)

            if _api.exchange.warmup_flag(st):
                return

            oat = _api.exchange.get_variable_value(st, global_vh[0]["oat"])
            sky_t = _api.exchange.get_variable_value(st, global_vh[0]["sky_temp"])
            for surf in act_handles[0]:
                vh = var_handles[0][surf]
                _data.append({
                    "month": month, "day": day, "hour": hour,
                    "surface": surf,
                    "oat_C": oat, "sky_temp_C": sky_t,
                    "surf_temp_C": _api.exchange.get_variable_value(st, vh["surf_temp"]),
                    "lw_rad_Wm2": _api.exchange.get_variable_value(st, vh["lw_rad"]),
                    "enet_applied_Wm2": enet_now,
                    "run": "csv_enet",
                })

        return _cb

    _api.runtime.callback_begin_zone_timestep_after_init_heat_balance(_state, _make_callback())

    args = ["-d", _run_dir, "-w", epw_file, idf_file]
    if design_day_only:
        args = ["-D"] + args

    print(f"  Running with CSV Enet schedule ({len(enet_schedule)} rows)...", end=" ", flush=True)
    ret = _api.runtime.run_energyplus(_state, args)
    _api.state_manager.delete_state(_state)
    print(f"{'OK' if ret == 0 else 'FAILED'} ({len(_data)} pts)")

    return _data


# Example usage (uncomment and edit):
# csv_data = run_with_csv_enet(IDF_FILE, EPW_FILE, target_surfaces, "my_enet_schedule.csv")
# csv_df = pd.DataFrame(csv_data)
# csv_df.plot(x="hour", y=["surf_temp_C", "enet_applied_Wm2"], secondary_y="enet_applied_Wm2")